# Dr-Zyren


### 01 | Runtime


In [1]:
import os, sys, platform, subprocess, textwrap, json, time, uuid, shutil, sqlite3, asyncio, logging, re, difflib, mimetypes
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
except Exception as e:
    print("PyTorch check:", repr(e))

print("Text model access uses Cohere's HTTPS Chat API; no Node.js runtime is required.")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56
Text model access uses Cohere's HTTPS Chat API; no Node.js runtime is required.


### 02 | Install


In [2]:

%pip install --upgrade \
    "python-telegram-bot>=21.10,<23" \
    "httpx>=0.27,<1" \
    "faster-whisper==1.2.1" \
    "ctranslate2>=4.6,<5" \
    "av==18.1.0" \
    "transformers==5.23.0" \
    "huggingface-hub>=1.32,<3" \
    "accelerate>=1.2.1" \
    "bitsandbytes>=0.45.0" \
    "qwen-vl-utils>=0.0.10" \
    "sentencepiece" \
    "gradio>=6.31,<7" \
    "protobuf>=5.29.1,<7" \
    "Pillow>=10,<13"

print("Installation completed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 40.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 58.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 128.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.4/769.4 kB 41.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 65.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 48.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 53.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 100.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 101.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 39.9 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 52.4 MB/s  0:00:00
  Attempting uninstall: sentencepiece
    Found existing installation: sentencepiece 0.2.2
    Uninstalling sentencepiece-0.2.2:
      Successfully uninstalled sen

In [3]:
%pip uninstall -y Pillow
%pip install --no-cache-dir --force-reinstall "Pillow==11.3.0"

Found existing installation: pillow 12.3.0
Uninstalling pillow-12.3.0:
  Successfully uninstalled pillow-12.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 23.5 MB/s  0:00:00


In [4]:

import av
import gradio
import huggingface_hub
import transformers
from faster_whisper import WhisperModel
from telegram import Update
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    CLIPModel,
    CLIPProcessor
)

print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("PyAV:", av.__version__)
print("Gradio:", gradio.__version__)
print("All imports successful!")

Transformers: 5.23.0
Hugging Face Hub: 2.4.0
PyAV: 18.1.0
Gradio: 6.32.0
All imports successful!


### 03 | Drive


In [7]:
from google.colab import drive
from google.colab import userdata
from IPython.display import display

DRIVE_MOUNTED = False
try:
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
except Exception as e:
    print("Google Drive mount was not completed:", repr(e))

if DRIVE_MOUNTED:
    PROJECT_DIR = Path('/content/drive/MyDrive/Dr-Zyren')
else:
    PROJECT_DIR = Path('/content/Dr-Zyren')

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PROJECT_DIR / 'data'
UPLOAD_DIR = DATA_DIR / 'uploads'
CASE_IMAGE_DIR = DATA_DIR / 'reference_cases'
MODEL_CACHE_DIR = PROJECT_DIR / 'model_cache'
for p in (DATA_DIR, UPLOAD_DIR, CASE_IMAGE_DIR, MODEL_CACHE_DIR):
    p.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / 'dr_zyren.sqlite3'
print("Project data directory:", PROJECT_DIR)
print("Persistent Drive storage:", DRIVE_MOUNTED)

Mounted at /content/drive
Project data directory: /content/drive/MyDrive/Dr-Zyren
Persistent Drive storage: True


### 04 | Setup


In [9]:

from getpass import getpass

# Load Colab Secrets
try:
    from google.colab import userdata
except ImportError:
    userdata = None


# Cohere API configuration
COHERE_API_URL = "https://api.cohere.com/v2/chat"

COHERE_MODELS = {
    "1": {
        "name": "Command A+ (recommended)",
        "id": "command-a-plus-05-2026",
        "description": (
            "Newer multimodal model: reasoning + images + "
            "agentic tasks; 48 languages; 128K context."
        ),
    },
    "2": {
        "name": "Command A Reasoning",
        "id": "command-a-reasoning-08-2025",
        "description": (
            "Text reasoning; 23 languages; 256K context. "
            "Images use the local Qwen vision model."
        ),
    },
}


# Choose primary model
print("Choose the primary text model:")

for key, option in COHERE_MODELS.items():
    print(f"{key}. {option['name']} — {option['description']}")

model_choice = input("Model [1]: ").strip() or "1"

if model_choice not in COHERE_MODELS:
    raise ValueError("Choose 1 or 2 for the Cohere model.")

COHERE_MODEL_NAME = COHERE_MODELS[model_choice]["name"]
COHERE_MODEL_ID = COHERE_MODELS[model_choice]["id"]


# Read Cohere API key
COHERE_API_KEY = ""

if userdata is not None:
    try:
        COHERE_API_KEY = userdata.get("COHERE_API_KEY") or ""
    except Exception:
        pass

if not COHERE_API_KEY:
    COHERE_API_KEY = getpass(
        "Paste your Cohere Trial API Key (input hidden): "
    ).strip()

if not COHERE_API_KEY:
    raise ValueError(
        "Cohere API key is required. "
        "Create a key at https://dashboard.cohere.com/api-keys"
    )


# Read Telegram Bot Token
TELEGRAM_BOT_TOKEN = ""

if userdata is not None:
    try:
        TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN") or ""
    except Exception:
        pass

if not TELEGRAM_BOT_TOKEN:
    TELEGRAM_BOT_TOKEN = getpass(
        "Paste your Telegram Bot Token (input hidden): "
    ).strip()

if not TELEGRAM_BOT_TOKEN:
    raise ValueError("Telegram Bot Token is required.")


# Project settings
ALLOWED_USER_IDS = set()

MAX_TELEGRAM_FILE_MB = 18
MAX_CONTEXT_CHARS = 24000
RECENT_MESSAGES_TO_SEND = 18


# Confirmation
print("\nSetup completed successfully.")
print("Telegram token loaded: yes")
print("Cohere API key loaded: yes (value hidden)")

print(
    "Selected model:",
    COHERE_MODEL_NAME,
    "[",
    COHERE_MODEL_ID,
    "]"
)

print("Model fallback: DISABLED")

Choose the primary text model:
1. Command A+ (recommended) — Newer multimodal model: reasoning + images + agentic tasks; 48 languages; 128K context.
2. Command A Reasoning — Text reasoning; 23 languages; 256K context. Images use the local Qwen vision model.
Model [1]: 1

Setup completed successfully.
Telegram token loaded: yes
Cohere API key loaded: yes (value hidden)
Selected model: Command A+ (recommended) [ command-a-plus-05-2026 ]
Model fallback: DISABLED


### 05 | Cohere


In [10]:
import asyncio
import httpx
import json
import base64
import io
from PIL import Image, ImageOps

class CohereAPIError(RuntimeError):
    pass


def image_data_url(image_path):
    path = Path(image_path)
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")
        image.thumbnail((2048, 2048))
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=88, optimize=True)
    raw = buffer.getvalue()
    if len(raw) > 12 * 1024 * 1024:
        raise ValueError("الصورة أكبر من الحد المسموح بعد الضغط.")
    return "data:image/jpeg;base64," + base64.b64encode(raw).decode("ascii")


class CohereChatClient:
    def __init__(self, api_key, model_id, endpoint=COHERE_API_URL):
        if not api_key:
            raise ValueError("COHERE_API_KEY is missing.")
        self.api_key = api_key
        self.model_id = model_id
        self.endpoint = endpoint
        self.status = "not_tested"
        self.last_endpoint = endpoint
        self._semaphore = asyncio.Semaphore(2)
        self._client = self._new_http_client()

    def _new_http_client(self):
        return httpx.AsyncClient(
            timeout=httpx.Timeout(120.0, connect=20.0),
            headers={
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json",
                "Accept": "application/json",
                "X-Client-Name": "Dr-Zyren-Colab",
            },
        )

    def _ensure_open(self):
        if self._client is None or self._client.is_closed:
            self._client = self._new_http_client()
            self._semaphore = asyncio.Semaphore(2)
            self.status = "ready"

    async def ask(self, chat_key, prompt, timeout_seconds=110, image_paths=None):
        self._ensure_open()
        image_paths = list(image_paths or [])
        if image_paths and self.model_id != "command-a-plus-05-2026":
            raise CohereAPIError("الموديل المختار نصّي فقط؛ استخدم مسار الرؤية المحلي لهذه الصورة.")

        content = [{"type": "text", "text": str(prompt)}]
        for image_path in image_paths[:4]:
            content.append({
                "type": "image_url",
                "image_url": {"url": image_data_url(image_path), "detail": "high"},
            })
        message_content = content if image_paths else str(prompt)
        payload = {
            "model": self.model_id,
            "messages": [{"role": "user", "content": message_content}],
            "max_tokens": 4096,
        }
        timeout = httpx.Timeout(max(15.0, min(float(timeout_seconds or 90), 120.0)), connect=20.0)
        async with self._semaphore:
            try:
                response = await self._client.post(self.endpoint, json=payload, timeout=timeout)
            except httpx.TimeoutException as exc:
                self.status = "timeout"
                raise CohereAPIError("Cohere request timed out.") from exc
            except httpx.HTTPError as exc:
                self.status = "network_error"
                raise CohereAPIError(f"Could not reach Cohere: {type(exc).__name__}.") from exc

            if response.status_code >= 400:
                self.status = "api_error"
                try:
                    error_data = response.json()
                    detail = error_data.get("message") or error_data.get("error") or ""
                except Exception:
                    detail = ""
                if response.status_code == 429:
                    detail = (str(detail) + " | قد يكون حد الاستخدام المجاني قد انتهى.").strip(" |")
                if response.status_code in (401, 403):
                    detail = "مفتاح Cohere غير مقبول أو لا يملك صلاحية لهذا الموديل."
                raise CohereAPIError(f"Cohere HTTP {response.status_code}: {str(detail or 'Request failed')[:700]}")

            try:
                data = response.json()
            except ValueError as exc:
                self.status = "parse_error"
                raise CohereAPIError("Cohere returned invalid JSON.") from exc

            message = data.get("message") or {}
            content = message.get("content") or []
            if isinstance(content, str):
                answer = content.strip()
            else:
                answer = "\n".join(
                    item.get("text", "").strip()
                    for item in content
                    if isinstance(item, dict) and item.get("type") == "text" and item.get("text", "").strip()
                ).strip()
            if not answer:
                self.status = "empty_response"
                raise CohereAPIError(f"Cohere response contained no text (finish_reason={data.get('finish_reason', 'unknown')}).")
            self.status = "connected"
            return answer

    async def close(self):
        if self._client and not self._client.is_closed:
            await self._client.aclose()
        self.status = "stopped"


model_client = CohereChatClient(COHERE_API_KEY, COHERE_MODEL_ID)
print("Cohere ready:", COHERE_MODEL_ID)

Cohere ready: command-a-plus-05-2026


### 06 | Instructions


In [11]:
THIRD_CONTRIBUTOR_NAME = ""  # اكتب الاسم هنا عندما يتأكد.
DR_ZYREN_TEAM = ["Mohamed Abdelnaser (محمد عبدالناصر)", "Farah Mohamed (فرح محمد)","Basmala Hossam (بسملة حسام)"]
if THIRD_CONTRIBUTOR_NAME.strip():
    DR_ZYREN_TEAM.append(THIRD_CONTRIBUTOR_NAME.strip())

DR_ZYREN_SYSTEM_PROMPT = f"""
أنت Dr-Zyren، مساعد معلومات طبية ودعم قرار سريري. تواصل بالعربية المصرية الواضحة إذا كتب المستخدم بالعربية، وباللغة التي يستخدمها إذا كانت غير ذلك.

الدقة الطبية:
- لا تدّعِ فحص المريض أو تأكيد تشخيص من صورة أو من حالات مشابهة.
- افصل بين ما يظهر فعلاً، والاحتمالات، وما لا نعرفه، والخطوة التالية.
- لا تخترع تاريخاً مرضياً أو نتائج تحاليل أو مراجع. لا تعطِ جرعات أو وصفات خطرة أو غير مؤكدة.
- اسأل سؤالاً واحداً أو اثنين فقط عندما تكون معلومة أساسية ناقصة.
- الصور قد تكون غير كافية أو غير واضحة؛ اطلب صورة أو فحصاً مناسباً عند الحاجة.
- الحالات المرجعية أمثلة للمقارنة وليست تشخيصاً، والتشابه البصري لا يثبت تطابق المرض.

الطوارئ:
- إذا ظهرت علامات خطر مثل ألم/ضغط صدري شديد، ضيق نفس شديد، فقدان وعي، تشنج مستمر، نزيف لا يتوقف، أعراض جلطة مفاجئة، حساسية شديدة مع تورم/اختناق، أو تدهور سريع: ابدأ الرد بكلمة «طوارئ الآن» بوضوح.
- اطلب الاتصال فوراً بالإسعاف في مصر 123 أو الذهاب لأقرب طوارئ. لا تؤخر طلب المساعدة لاستكمال المحادثة.
- لا تستخدم تحذير الطوارئ لمجرد أعراض بسيطة بلا مؤشرات خطر. خارج مصر اطلب رقم الطوارئ المحلي.

طريقة الرد:
- كن مختصراً وطبيعياً ومباشراً. ابدأ بالخلاصة، ثم أهم سبب أو احتمال، ثم الخطوة العملية.
- لا تستخدم عناوين كثيرة أو مقدمات طويلة أو قوائم طويلة. استخدم نقاطاً قليلة عند الحاجة فقط.
- لا تكرر السؤال ولا تعيد صياغة كلام المستخدم بلا فائدة. صحّح العربية واجعلها سليمة وسهلة القراءة.
- لا تدّعِ اليقين. اذكر متى يلزم طبيب أو فحص حضوري.

هوية Dr-Zyren:
- إذا سُئلت عن الجهة التعليمية، قل: Innovation University (جامعة ابتكار).
- إذا سُئلت من أعدّ أو خصّص Dr-Zyren، قل إن إعداد وتخصيص التطبيق تم بواسطة: {" و".join(DR_ZYREN_TEAM)}.
- لا تقل إن هؤلاء الأشخاص درّبوا أوزان نموذج Cohere الأساسية؛ مزوّد النموذج هو Cohere. فرّق بين إعداد التطبيق وتدريب النموذج الأساسي.
- لا تخترع اسماً ثالثاً. أضفه فقط بعد إدخاله في THIRD_CONTRIBUTOR_NAME.

هذا نموذج دعم ومعلومات تجريبي، وليس بديلاً عن التقييم الطبي أو الطبيب المعالج.
""".strip()
print("Instructions ready.")

Instructions ready.


### 07 | Database


In [12]:
from datetime import datetime, timezone

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def get_db():
    conn = sqlite3.connect(DB_PATH, timeout=30)
    conn.row_factory = sqlite3.Row
    return conn

with get_db() as conn:
    conn.executescript("""
    PRAGMA journal_mode=WAL;
    CREATE TABLE IF NOT EXISTS conversations (
        conversation_id TEXT PRIMARY KEY,
        telegram_user_id TEXT NOT NULL,
        created_at TEXT NOT NULL,
        is_active INTEGER NOT NULL DEFAULT 1,
        summary TEXT NOT NULL DEFAULT '',
        summary_upto_message_id INTEGER NOT NULL DEFAULT 0
    );
    CREATE INDEX IF NOT EXISTS idx_conversations_user ON conversations(telegram_user_id, is_active);
    CREATE TABLE IF NOT EXISTS messages (
        message_id INTEGER PRIMARY KEY AUTOINCREMENT,
        conversation_id TEXT NOT NULL,
        telegram_user_id TEXT NOT NULL,
        role TEXT NOT NULL,
        content TEXT NOT NULL,
        created_at TEXT NOT NULL
    );
    CREATE INDEX IF NOT EXISTS idx_messages_conv ON messages(conversation_id, message_id);
    CREATE TABLE IF NOT EXISTS reference_cases (
        case_id TEXT PRIMARY KEY,
        image_path TEXT NOT NULL,
        description TEXT NOT NULL,
        findings TEXT NOT NULL,
        reference_explanation TEXT NOT NULL,
        category TEXT NOT NULL,
        embedding BLOB,
        created_at TEXT NOT NULL
    );
    """)
print("SQLite database ready:", DB_PATH)

SQLite database ready: /content/drive/MyDrive/Dr-Zyren/data/dr_zyren.sqlite3


### 08 | Memory


In [14]:
def get_active_conversation(user_id):
    user_id = str(user_id)
    with get_db() as conn:
        row = conn.execute("SELECT conversation_id FROM conversations WHERE telegram_user_id=? AND is_active=1 ORDER BY created_at DESC LIMIT 1", (user_id,)).fetchone()
        if row:
            return row["conversation_id"]
        cid = str(uuid.uuid4())
        conn.execute("UPDATE conversations SET is_active=0 WHERE telegram_user_id=?", (user_id,))
        conn.execute("INSERT INTO conversations(conversation_id,telegram_user_id,created_at,is_active) VALUES(?,?,?,1)", (cid, user_id, utc_now()))
        return cid

def create_new_conversation(user_id):
    user_id = str(user_id)
    cid = str(uuid.uuid4())
    with get_db() as conn:
        conn.execute("UPDATE conversations SET is_active=0 WHERE telegram_user_id=?", (user_id,))
        conn.execute("INSERT INTO conversations(conversation_id,telegram_user_id,created_at,is_active) VALUES(?,?,?,1)", (cid, user_id, utc_now()))
    return cid

def add_message(user_id, conversation_id, role, content):
    with get_db() as conn:
        cur = conn.execute("INSERT INTO messages(conversation_id,telegram_user_id,role,content,created_at) VALUES(?,?,?,?,?)", (conversation_id, str(user_id), role, str(content), utc_now()))
        return cur.lastrowid

def read_messages(conversation_id):
    with get_db() as conn:
        rows = conn.execute("SELECT message_id,role,content,created_at FROM messages WHERE conversation_id=? ORDER BY message_id", (conversation_id,)).fetchall()
    return [dict(r) for r in rows]

def get_conversation_summary(conversation_id):
    with get_db() as conn:
        row = conn.execute("SELECT summary,summary_upto_message_id FROM conversations WHERE conversation_id=?", (conversation_id,)).fetchone()
    return (row["summary"], row["summary_upto_message_id"]) if row else ("", 0)

def save_conversation_summary(conversation_id, summary, upto_id):
    with get_db() as conn:
        conn.execute("UPDATE conversations SET summary=?,summary_upto_message_id=? WHERE conversation_id=?", (summary, int(upto_id), conversation_id))

def clear_active_conversation(user_id):
    cid = get_active_conversation(user_id)
    with get_db() as conn:
        conn.execute("DELETE FROM messages WHERE conversation_id=?", (cid,))
        conn.execute("UPDATE conversations SET summary='',summary_upto_message_id=0 WHERE conversation_id=?", (cid,))
    return cid

def format_history(messages, max_chars=MAX_CONTEXT_CHARS):
    lines=[]
    for m in messages:
        label = "User" if m["role"] == "user" else "Dr-Zyren"
        lines.append(f"{label}: {m['content']}")
    text = "\n".join(lines)
    if len(text) > max_chars:
        text = "[Older conversation content omitted from this request because of the context limit. The complete transcript remains stored in SQLite.]\n" + text[-max_chars:]
    return text

print("Conversation memory functions ready.")

Conversation memory functions ready.


In [15]:
%pip install -q "faster-whisper>=1.1.0"

from faster_whisper import WhisperModel
from importlib.metadata import version

print("faster-whisper:", version("faster-whisper"))
print("Whisper import successful!")

faster-whisper: 1.2.1
Whisper import successful!


In [16]:
%pip install -q --force-reinstall "av==18.1.0"

import av
print("PyAV version:", av.__version__)

PyAV version: 18.1.0


In [17]:
from pathlib import Path

# Move model cache outside Google Drive
MODEL_CACHE_DIR = Path("/content/Dr-Zyren-model-cache")
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Reset Whisper loader
whisper_model = None

print("New model cache:", MODEL_CACHE_DIR)
print("Whisper cache path:", MODEL_CACHE_DIR / "faster-whisper")

New model cache: /content/Dr-Zyren-model-cache
Whisper cache path: /content/Dr-Zyren-model-cache/faster-whisper


### 09 | Whisper


In [18]:
from faster_whisper import WhisperModel

WHISPER_MODEL_SIZE = "small"  # Change to "medium" or "large-v3" if you have enough runtime resources.
WHISPER_DEVICE = "cpu"         # Keeps GPU memory available for the vision/reasoning model.
WHISPER_COMPUTE_TYPE = "int8"
whisper_model = None

def load_whisper():
    global whisper_model
    if whisper_model is None:
        print(f"Downloading/loading faster-whisper {WHISPER_MODEL_SIZE}...")
        whisper_model = WhisperModel(
            WHISPER_MODEL_SIZE,
            device=WHISPER_DEVICE,
            compute_type=WHISPER_COMPUTE_TYPE,
            download_root=str(MODEL_CACHE_DIR / "faster-whisper")
        )
    return whisper_model

def transcribe_audio(audio_path):
    model = load_whisper()
    segments, info = model.transcribe(
        str(audio_path),
        beam_size=5,
        vad_filter=True,
        condition_on_previous_text=False,
        task="transcribe"
    )
    text = " ".join(seg.text.strip() for seg in segments if seg.text.strip()).strip()
    return {"text": text, "language": getattr(info, "language", None), "language_probability": getattr(info, "language_probability", None)}

print("Whisper functions ready. Model will download the first time voice transcription is needed.")

Whisper functions ready. Model will download the first time voice transcription is needed.


### 10 | Vision model


In [19]:
import torch
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration
)

VISION_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

vision_model = None
vision_processor = None
VISION_MODEL_LOAD_ERROR = None

def load_vision_model():
    global vision_model, vision_processor, VISION_MODEL_LOAD_ERROR

    if vision_model is not None:
        return vision_model, vision_processor

    # Never attempt to load this 3B model on system RAM.
    if not torch.cuda.is_available():
        VISION_MODEL_LOAD_ERROR = (
            "GPU is unavailable. Qwen2.5-VL loading on CPU is disabled "
            "to prevent Colab RAM crashes."
        )
        raise RuntimeError(VISION_MODEL_LOAD_ERROR)

    try:
        free_memory, total_memory = torch.cuda.mem_get_info(0)

        print("GPU:", torch.cuda.get_device_name(0))
        print(f"Free GPU memory: {free_memory / 1024**3:.2f} GB")

        if free_memory < 5 * 1024**3:
            raise RuntimeError(
                "Insufficient free GPU memory. "
                "At least 5 GB free GPU memory is required."
            )

        print("Loading:", VISION_MODEL_ID)

        vision_processor = AutoProcessor.from_pretrained(
            VISION_MODEL_ID,
            cache_dir=str(MODEL_CACHE_DIR / "huggingface"),
            min_pixels=256 * 28 * 28,
            max_pixels=512 * 28 * 28
        )

        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )

        vision_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            VISION_MODEL_ID,
            quantization_config=quant_config,
            device_map="auto",
            torch_dtype=torch.float16,
            cache_dir=str(MODEL_CACHE_DIR / "huggingface"),
            low_cpu_mem_usage=True
        )

        vision_model.eval()

        VISION_MODEL_LOAD_ERROR = None

        print("Qwen vision model loaded successfully.")

        return vision_model, vision_processor

    except Exception as e:
        vision_model = None
        vision_processor = None

        VISION_MODEL_LOAD_ERROR = (
            f"{type(e).__name__}: {e}"
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print("Vision model loading failed:", VISION_MODEL_LOAD_ERROR)
        raise

print("Safe vision model loader initialized.")
print("Qwen will only load when a GPU is available.")

Safe vision model loader initialized.
Qwen will only load when a GPU is available.


### 11 | Vision


In [20]:
from qwen_vl_utils import process_vision_info

IMAGE_ANALYSIS_INSTRUCTIONS = """
Describe only what is visibly present in this image.
Identify image type when possible, visible structures,
image quality limitations, and findings that may be relevant
to a clinician. Do not claim a definitive diagnosis.
Do not invent findings. Clearly separate observations from possibilities.
""".strip()


def _model_device(model):
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )


def analyze_medical_image(
    image_path,
    user_question="Please analyze this image carefully."
):

    model, processor = load_vision_model()

    messages = [{
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": str(image_path)
            },
            {
                "type": "text",
                "text": IMAGE_ANALYSIS_INSTRUCTIONS +
                "\nUser question: " + user_question
            }
        ]
    }]

    text_prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(_model_device(model))

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=750,
            do_sample=False
        )

    new_tokens = generated[:, inputs.input_ids.shape[1]:]

    return processor.batch_decode(
        new_tokens,
        skip_special_tokens=True
    )[0].strip()


print("Qwen image-analysis function ready. Text replies use the selected Cohere model.")

Qwen image-analysis function ready. Text replies use the selected Cohere model.


### 12 | Image index


In [21]:
import numpy as np
from PIL import Image, ImageOps
from transformers import CLIPModel, CLIPProcessor

CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
clip_model = None
clip_processor = None

def load_clip_model():
    global clip_model, clip_processor
    if clip_model is None:
        print("Loading public CLIP image embedding model (retrieval only, not diagnosis)...")
        clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID, cache_dir=str(MODEL_CACHE_DIR / "huggingface"))
        clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID, cache_dir=str(MODEL_CACHE_DIR / "huggingface"))
        clip_model.to("cpu").eval()
    return clip_model, clip_processor

def image_embedding(image_path):
    model, processor = load_clip_model()
    img = ImageOps.exif_transpose(Image.open(image_path)).convert("RGB")
    inputs = processor(images=img, return_tensors="pt")
    with torch.inference_mode():
        vector = model.get_image_features(**inputs)
    vector = torch.nn.functional.normalize(vector, dim=-1)
    return vector[0].cpu().numpy().astype("float32")

print("Reference-image embedding functions ready.")

Reference-image embedding functions ready.


### 13 | Case library


In [22]:
CASE_CONFIG_PATH = DATA_DIR / "case_search_config.json"
CASE_SIMILARITY_THRESHOLD = 0.82
CASE_SEARCH_TOP_K = 3
if CASE_CONFIG_PATH.exists():
    try:
        saved_config = json.loads(CASE_CONFIG_PATH.read_text(encoding="utf-8"))
        CASE_SIMILARITY_THRESHOLD = float(saved_config.get("similarity_threshold", CASE_SIMILARITY_THRESHOLD))
        CASE_SEARCH_TOP_K = int(saved_config.get("top_k", CASE_SEARCH_TOP_K))
    except Exception:
        pass

def add_reference_case(image_path, description, findings, reference_explanation, category="general"):
    src = Path(image_path)
    if not src.exists():
        raise FileNotFoundError(src)
    case_id = str(uuid.uuid4())[:12]
    dest = CASE_IMAGE_DIR / f"{case_id}{src.suffix.lower() or '.jpg'}"
    shutil.copy2(src, dest)
    emb = image_embedding(dest)
    with get_db() as conn:
        conn.execute(
            "INSERT INTO reference_cases(case_id,image_path,description,findings,reference_explanation,category,embedding,created_at) VALUES(?,?,?,?,?,?,?,?)",
            (case_id, str(dest), description.strip(), findings.strip(), reference_explanation.strip(), category.strip(), emb.tobytes(), utc_now())
        )
    return case_id

def list_reference_cases(limit=100):
    with get_db() as conn:
        rows = conn.execute("SELECT case_id,description,category,created_at FROM reference_cases ORDER BY created_at DESC LIMIT ?", (int(limit),)).fetchall()
    return [dict(r) for r in rows]

def search_reference_cases(image_path, top_k=None, threshold=None):
    top_k = CASE_SEARCH_TOP_K if top_k is None else int(top_k)
    threshold = CASE_SIMILARITY_THRESHOLD if threshold is None else float(threshold)
    with get_db() as conn:
        rows = conn.execute("SELECT * FROM reference_cases WHERE embedding IS NOT NULL").fetchall()
    if not rows:
        return []
    query = image_embedding(image_path)
    scored=[]
    for row in rows:
        vec = np.frombuffer(row["embedding"], dtype=np.float32)
        if vec.shape != query.shape:
            continue
        score = float(np.dot(query, vec) / (np.linalg.norm(query) * np.linalg.norm(vec) + 1e-8))
        if score >= threshold:
            scored.append((score, dict(row)))
    scored.sort(key=lambda x: x[0], reverse=True)
    results=[]
    for score, row in scored[:top_k]:
        results.append({"similarity": score, "case_id": row["case_id"], "description": row["description"], "findings": row["findings"], "reference_explanation": row["reference_explanation"], "category": row["category"]})
    return results

def delete_reference_case(case_id):
    with get_db() as conn:
        row = conn.execute("SELECT image_path FROM reference_cases WHERE case_id=?", (str(case_id),)).fetchone()
        if not row:
            return False
        conn.execute("DELETE FROM reference_cases WHERE case_id=?", (str(case_id),))
    try: Path(row["image_path"]).unlink(missing_ok=True)
    except Exception: pass
    return True

print("Case library ready.")

Case library ready.


### 14 | Add cases


In [23]:
import gradio as gr
import csv

CASE_MANIFEST_TEMPLATE = PROJECT_DIR / "case_manifest_template.csv"
with open(CASE_MANIFEST_TEMPLATE, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["image", "category", "description", "findings", "reference_explanation"])
    writer.writeheader()
    writer.writerow({"image": "sample.jpg", "category": "dental", "description": "Case description", "findings": "Verified findings", "reference_explanation": "Verified reference explanation"})


def add_case_batch(image_files, manifest_file, category, description, findings, explanation, threshold, top_k):
    global CASE_SIMILARITY_THRESHOLD, CASE_SEARCH_TOP_K
    if not image_files:
        return "ارفع صورة واحدة على الأقل."
    CASE_SIMILARITY_THRESHOLD = float(threshold)
    CASE_SEARCH_TOP_K = int(top_k)
    CASE_CONFIG_PATH.write_text(json.dumps({
        "similarity_threshold": CASE_SIMILARITY_THRESHOLD,
        "top_k": CASE_SEARCH_TOP_K
    }, ensure_ascii=False, indent=2), encoding="utf-8")

    manifest = {}
    if manifest_file:
        with open(manifest_file, newline="", encoding="utf-8-sig") as f:
            for row in csv.DictReader(f):
                name = (row.get("image") or "").strip()
                if name:
                    manifest[Path(name).name] = row

    saved, skipped = [], []
    for item in image_files:
        image_path = item if isinstance(item, str) else getattr(item, "name", str(item))
        name = Path(image_path).name
        row = manifest.get(name, {})
        cat = (row.get("category") or category or "general").strip()
        desc = (row.get("description") or description or "").strip()
        fnd = (row.get("findings") or findings or "").strip()
        ref = (row.get("reference_explanation") or explanation or "").strip()
        if not desc or not fnd or not ref:
            skipped.append(name + " (بيانات الحالة ناقصة)")
            continue
        try:
            saved.append(add_reference_case(image_path, desc, fnd, ref, cat))
        except Exception as e:
            skipped.append(name + " (" + type(e).__name__ + ")")

    result = f"تمت إضافة {len(saved)} حالة."
    if skipped:
        result += "\nلم تتم إضافة: " + "، ".join(skipped[:10])
    result += f"\nحد التشابه: {CASE_SIMILARITY_THRESHOLD:.2f} | عدد النتائج: {CASE_SEARCH_TOP_K}"
    result += "\nالحالات تُستخدم للبحث عن أمثلة مشابهة، ولا تغيّر أوزان الموديل أو تثبت التشخيص."
    return result

with gr.Blocks(title="Dr-Zyren Case Library") as teaching_app:
    gr.Markdown("## مكتبة الحالات")
    gr.Markdown("ارفع صوراً مجهولة الهوية. التشابه يساعد في استرجاع أمثلة قريبة فقط، ولا يؤكد التشخيص.")
    case_files = gr.File(label="صور الحالات", file_count="multiple", file_types=["image"], type="filepath")
    manifest = gr.File(label="ملف CSV اختياري لبيانات كل صورة", file_types=[".csv"], type="filepath")
    gr.File(value=str(CASE_MANIFEST_TEMPLATE), label="نموذج CSV للتعبئة", interactive=False)
    with gr.Row():
        case_category = gr.Textbox(label="التخصص / التصنيف", placeholder="dental")
        case_description = gr.Textbox(label="وصف الحالة")
    case_findings = gr.Textbox(label="النتائج المؤكدة", lines=2)
    case_explanation = gr.Textbox(label="الشرح المرجعي", lines=2)
    with gr.Row():
        similarity = gr.Slider(0.50, 0.99, value=0.82, step=0.01, label="حد التشابه")
        top_k = gr.Slider(1, 8, value=3, step=1, label="عدد الحالات المقترحة")
    add_button = gr.Button("إضافة الحالات وبناء الفهرس", variant="primary")
    case_result = gr.Textbox(label="النتيجة", lines=4)
    add_button.click(add_case_batch, [case_files, manifest, case_category, case_description, case_findings, case_explanation, similarity, top_k], case_result)

print("Case library ready.")
teaching_app.launch(share=False, inline=True)

Case library ready.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

### 15 | Context


In [24]:
async def summarize_old_messages(conversation_id, user_id):

    messages = read_messages(conversation_id)

    summary, upto_id = get_conversation_summary(conversation_id)

    cutoff_messages = messages[:-RECENT_MESSAGES_TO_SEND]

    unsummarized_old = [
        m for m in cutoff_messages
        if m["message_id"] > upto_id
    ]

    if len(unsummarized_old) < 24:
        return summary

    transcript = format_history(
        unsummarized_old,
        max_chars=18000
    )

    prompt = f"""Update this longitudinal conversation memory.
Preserve clinically relevant facts, symptom onset/duration,
laterality, severity, relevant medical/dental history,
medications/allergies mentioned, test results, prior advice,
unresolved questions, and explicit corrections.
Do not invent facts. Keep it concise and clearly mark unknowns.

Previous memory:
{summary or '(none)'}

New conversation turns to incorporate:
{transcript}

Return only the updated memory."""

    try:

        new_summary = await model_client.ask(
            f"memory-{user_id}",
            prompt,
            timeout_seconds=110
        )

        if not new_summary or len(new_summary.strip()) < 10:
            raise CohereAPIError(
                "Cohere returned an empty/too-short memory summary."
            )

        upto = unsummarized_old[-1]["message_id"]

        save_conversation_summary(
            conversation_id,
            new_summary.strip(),
            upto
        )

        return new_summary.strip()

    except Exception as e:

        print(
            "Cohere memory summarization unavailable:",
            str(e)[:300]
        )

        return summary


async def build_model_prompt(
    user_id,
    conversation_id,
    current_input,
    extra_context=""
):

    all_messages = read_messages(conversation_id)

    summary, _ = get_conversation_summary(conversation_id)

    if len(all_messages) > RECENT_MESSAGES_TO_SEND + 8:
        summary = await summarize_old_messages(
            conversation_id,
            user_id
        ) or summary

    history_messages = all_messages

    if (
        history_messages
        and history_messages[-1]["role"] == "user"
        and history_messages[-1]["content"] == current_input
    ):
        history_messages = history_messages[:-1]

    recent = history_messages[-RECENT_MESSAGES_TO_SEND:]

    history_text = format_history(
        recent,
        max_chars=MAX_CONTEXT_CHARS
    )

    return f"""{DR_ZYREN_SYSTEM_PROMPT}

LONGITUDINAL MEMORY SUMMARY:
{summary or '(No saved summary yet.)'}

RECENT CONVERSATION:
{history_text}

CURRENT USER INPUT:
{current_input}

ADDITIONAL PROCESSING CONTEXT:
{extra_context or '(None)'}

Respond to the current user while respecting all relevant earlier context."""


print("Memory builder ready. Summaries use the selected Cohere model.")

Memory builder ready. Summaries use the selected Cohere model.


### 16 | Reply


In [25]:
async def get_final_answer(
    user_id,
    conversation_id,
    current_input,
    extra_context="",
    image_paths=None
):

    prompt = await build_model_prompt(
        user_id,
        conversation_id,
        current_input,
        extra_context
    )

    try:

        answer = await model_client.ask(
            str(user_id),
            prompt,
            timeout_seconds=110,
            image_paths=image_paths
        )

        if answer and len(answer.strip()) >= 2:
            return answer.strip(), "cohere_api"

        raise CohereAPIError(
            "The selected Cohere model returned an empty response."
        )

    except Exception as web_error:

        print(
            "Cohere model unavailable:",
            str(web_error)[:700]
        )

        return (
            "تعذر الوصول إلى موديل Cohere المختار. "
            "لم يتم استخدام أي موديل نصي بديل. "
            "يرجى التحقق من مفتاح Cohere وحدود الاستخدام ثم إعادة المحاولة.\n\n"
            + str(web_error)[:500],
            "unavailable"
        )


print("Response pipeline ready: selected Cohere model only; no silent fallback.")

Response pipeline ready: selected Cohere model only; no silent fallback.


### 17 | Telegram


In [26]:
%pip install -q "python-telegram-bot>=21.10,<23"

from telegram import Update
from telegram.ext import Application

print("Telegram library installed successfully!")

Telegram library installed successfully!


In [27]:
from telegram import Update
from telegram.constants import ChatAction
from telegram.ext import Application, CommandHandler, MessageHandler, ContextTypes, filters
from collections import defaultdict

user_locks = defaultdict(asyncio.Lock)

def user_is_allowed(user_id):
    if not ALLOWED_USER_IDS:
        return True
    return int(user_id) in ALLOWED_USER_IDS

async def ensure_private_authorized(update):
    if not update.effective_user or not update.effective_chat:
        return False
    if update.effective_chat.type != "private":
        await update.effective_message.reply_text("Please use Dr-Zyren in a private chat.")
        return False
    if not user_is_allowed(update.effective_user.id):
        await update.effective_message.reply_text("This bot is currently restricted to authorized testers.")
        return False
    return True

async def send_long_text(message, text):
    text = str(text or "No response was generated.")
    limit = 3900
    while text:
        chunk = text[:limit]
        if len(text) > limit:
            split_at = chunk.rfind("\n")
            if split_at > 1000:
                chunk = chunk[:split_at]
        await message.reply_text(chunk)
        text = text[len(chunk):].lstrip()

async def send_typing(context, chat_id):
    try:
        await context.bot.send_chat_action(chat_id=chat_id, action=ChatAction.TYPING)
    except Exception:
        pass

async def download_telegram_file(file_obj, suffix):
    if file_obj.file_size and file_obj.file_size > MAX_TELEGRAM_FILE_MB * 1024 * 1024:
        raise ValueError(f"File is larger than the configured {MAX_TELEGRAM_FILE_MB} MB limit.")
    dest = UPLOAD_DIR / f"tg_{uuid.uuid4().hex}{suffix}"
    await file_obj.download_to_drive(custom_path=str(dest))
    return dest

print("Telegram helper functions ready.")

Telegram helper functions ready.


### 18 | Commands


In [28]:
async def start_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not await ensure_private_authorized(update): return
    get_active_conversation(update.effective_user.id)
    await update.message.reply_text(
        "أهلاً بك في Dr-Zyren. ابعت سؤالك كتابة أو صورة أو تسجيل صوتي.\n"
        "/newchat محادثة جديدة | /history السجل | /clear مسح المحادثة | /myid رقمك"
    )

async def help_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not await ensure_private_authorized(update): return
    await update.message.reply_text("ابعت نص أو صورة أو تسجيل صوتي. الأوامر: /newchat /history /clear /myid")

async def myid_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not await ensure_private_authorized(update): return
    await update.message.reply_text(f"Telegram user ID: {update.effective_user.id}")

async def newchat_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not await ensure_private_authorized(update): return
    cid = create_new_conversation(update.effective_user.id)
    await update.message.reply_text("بدأنا محادثة جديدة. المحادثة السابقة محفوظة.\nID: " + cid[:8])

async def clear_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not await ensure_private_authorized(update): return
    clear_active_conversation(update.effective_user.id)
    await update.message.reply_text("تم مسح المحادثة الحالية فقط.")

async def history_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not await ensure_private_authorized(update): return
    cid = get_active_conversation(update.effective_user.id)
    msgs = read_messages(cid)[-12:]
    if not msgs:
        await update.message.reply_text("لا توجد رسائل بعد.")
        return
    lines = [f"{'You' if m['role']=='user' else 'Dr-Zyren'}: {m['content'][:450]}" for m in msgs]
    await send_long_text(update.message, "\n\n".join(lines))

print("Commands ready.")

Commands ready.


### 19 | Telegram input


In [29]:
async def process_user_input(update, context, user_text, image_path=None, audio_transcript=None):
    user_id = update.effective_user.id
    chat_id = update.effective_chat.id
    async with user_locks[user_id]:
        await send_typing(context, chat_id)
        status_message = await update.message.reply_text("براجع رسالتك…")
        try:
            conversation_id = get_active_conversation(user_id)
            current_parts = []
            if user_text:
                current_parts.append("User text: " + user_text)
            if audio_transcript:
                current_parts.append("Voice transcription: " + audio_transcript)
            extra_context_parts = []
            image_paths = []

            if image_path:
                if COHERE_MODEL_ID == "command-a-plus-05-2026":
                    current_parts.append("حلّل الصورة المرفقة مباشرة، واذكر حدود ما يمكن استنتاجه منها.")
                    image_paths = [image_path]
                else:
                    visual_findings = await asyncio.to_thread(analyze_medical_image, image_path, user_text or "حلل الصورة طبياً.")
                    current_parts.append("Visual observations from the local image model:\n" + visual_findings)
                    extra_context_parts.append("ملاحظات الصورة غير المؤكدة:\n" + visual_findings)

                try:
                    matches = await asyncio.to_thread(search_reference_cases, image_path)
                except Exception as case_error:
                    logging.warning("Case search unavailable: %s", type(case_error).__name__)
                    matches = []
                if matches:
                    formatted = [
                        f"حالة مشابهة (تشابه بصري {m['similarity']:.2f}) | {m['category']}\nالوصف: {m['description']}\nالنتائج المرجعية: {m['findings']}\nالشرح: {m['reference_explanation']}"
                        for m in matches
                    ]
                    extra_context_parts.append("أمثلة مرجعية للمقارنة فقط، وليست تشخيصاً مؤكداً:\n" + "\n\n".join(formatted))

            current_input = "\n\n".join(current_parts).strip() or "اسأل المستخدم عما يريد."
            add_message(user_id, conversation_id, "user", current_input)
            answer, _ = await get_final_answer(
                user_id, conversation_id, current_input,
                "\n\n".join(extra_context_parts), image_paths=image_paths
            )
            add_message(user_id, conversation_id, "assistant", answer)
            try: await status_message.delete()
            except Exception: pass
            await send_long_text(update.message, answer)
        except Exception as e:
            logging.exception("Dr-Zyren processing failed")
            try: await status_message.edit_text(f"حصل خطأ: {type(e).__name__}: {str(e)[:500]}")
            except Exception: await update.message.reply_text(f"حصل خطأ: {type(e).__name__}: {str(e)[:400]}")

async def text_handler(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not await ensure_private_authorized(update): return
    text = (update.message.text or "").strip()
    if text:
        await process_user_input(update, context, text)

async def image_handler(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not await ensure_private_authorized(update): return
    message = update.message
    caption = (message.caption or "").strip()
    try:
        if message.photo:
            tg_file = await message.photo[-1].get_file()
            path = await download_telegram_file(tg_file, ".jpg")
        elif message.document:
            tg_file = await message.document.get_file()
            suffix = Path(message.document.file_name or "image.png").suffix or ".png"
            path = await download_telegram_file(tg_file, suffix)
        else:
            return
        await process_user_input(update, context, caption or "حلل الصورة المرفقة.", image_path=path)
    except Exception as e:
        await update.message.reply_text(f"تعذر تحليل الصورة: {type(e).__name__}: {str(e)[:350]}")

async def audio_handler(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not await ensure_private_authorized(update): return
    message = update.message
    try:
        audio_obj = message.voice or message.audio or message.document
        if audio_obj is None: return
        file_name = getattr(audio_obj, "file_name", None) or "voice.ogg"
        suffix = Path(file_name).suffix or (".ogg" if message.voice else ".mp3")
        tg_file = await audio_obj.get_file()
        audio_path = await download_telegram_file(tg_file, suffix)
        await send_typing(context, update.effective_chat.id)
        transcript_result = await asyncio.to_thread(transcribe_audio, audio_path)
        transcript = transcript_result["text"]
        if not transcript:
            await update.message.reply_text("الصوت مش واضح كفاية. جرّب تسجيله تاني.")
            return
        await process_user_input(update, context, "", audio_transcript=transcript)
    except Exception as e:
        await update.message.reply_text(f"تعذر تفريغ التسجيل: {type(e).__name__}: {str(e)[:400]}")

print("Telegram text, image and voice input ready.")

Telegram text, image and voice input ready.


### 20 | Telegram app


In [30]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

application = Application.builder().token(TELEGRAM_BOT_TOKEN).build()
application.add_handler(CommandHandler("start", start_command))
application.add_handler(CommandHandler("help", help_command))
application.add_handler(CommandHandler("myid", myid_command))
application.add_handler(CommandHandler("newchat", newchat_command))
application.add_handler(CommandHandler("clear", clear_command))
application.add_handler(CommandHandler("history", history_command))
application.add_handler(MessageHandler(filters.PHOTO | filters.Document.IMAGE, image_handler))
application.add_handler(MessageHandler(filters.VOICE | filters.AUDIO | filters.Document.AUDIO, audio_handler))
application.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, text_handler))

print("Telegram application configured. No polling has started yet.")

Telegram application configured. No polling has started yet.


### 21 | Test


In [31]:
COHERE_MODEL_TEST_OK = False
EXPECTED_TEST_TEXT = "DR-ZYREN COHERE TEST OK"

try:
    test_result = await model_client.ask(
        "__diagnostic__",
        "Reply exactly with: " + EXPECTED_TEST_TEXT,
        timeout_seconds=90,
    )

    if EXPECTED_TEST_TEXT.casefold() in test_result.casefold():
        COHERE_MODEL_TEST_OK = True
        print("COHERE MODEL TEST: PASSED")
        print("Endpoint:", model_client.last_endpoint)
        print("Model:", COHERE_MODEL_ID)
        print("Verified response:", test_result[:500])
    else:
        print("COHERE MODEL TEST: FAILED")
        print("Unexpected response:")
        print(test_result[:700])

except Exception as e:
    COHERE_MODEL_TEST_OK = False
    print("COHERE MODEL TEST: FAILED")
    print(type(e).__name__ + ":", str(e)[:1000])

print("Telegram startup allowed:", COHERE_MODEL_TEST_OK)

COHERE MODEL TEST: PASSED
Endpoint: https://api.cohere.com/v2/chat
Model: command-a-plus-05-2026
Verified response: DR-ZYREN COHERE TEST OK
Telegram startup allowed: True


### 22 | Colab chat


In [32]:
import gradio as gr

COLAB_USER_ID = "colab-chat"
COLAB_CONVERSATION_ID = get_active_conversation(COLAB_USER_ID)

async def colab_chat_send(message, image_file, audio_file, history):
    global COLAB_CONVERSATION_ID
    history = list(history or [])
    text = (message or "").strip()
    transcript = ""
    if audio_file:
        result = await asyncio.to_thread(transcribe_audio, audio_file)
        transcript = result.get("text", "").strip()
    if not text and not transcript and not image_file:
        return history, "", None, None, "اكتب رسالة أو أرفق صورة/تسجيل."

    parts = []
    if text: parts.append(text)
    if transcript: parts.append("التسجيل الصوتي: " + transcript)
    image_paths = []
    extra = []
    if image_file:
        if COHERE_MODEL_ID == "command-a-plus-05-2026":
            parts.append("حلّل الصورة المرفقة مباشرة.")
            image_paths = [image_file]
        else:
            visual = await asyncio.to_thread(analyze_medical_image, image_file, text or "حلل الصورة.")
            parts.append("ملاحظات الصورة: " + visual)
        try:
            matches = await asyncio.to_thread(search_reference_cases, image_file)
            if matches:
                extra.append("حالات مشابهة للمقارنة فقط:\n" + "\n\n".join(
                    f"{m['category']} | {m['similarity']:.2f}\n{m['description']}\n{m['findings']}\n{m['reference_explanation']}"
                    for m in matches
                ))
        except Exception as e:
            logging.warning("Case search note: %s", type(e).__name__)

    current_input = "\n\n".join(parts)
    add_message(COLAB_USER_ID, COLAB_CONVERSATION_ID, "user", current_input)
    answer, _ = await get_final_answer(
        COLAB_USER_ID, COLAB_CONVERSATION_ID, current_input,
        "\n\n".join(extra), image_paths=image_paths
    )
    add_message(COLAB_USER_ID, COLAB_CONVERSATION_ID, "assistant", answer)
    visible = current_input
    if image_file: visible += "\n[صورة مرفقة]"
    if audio_file: visible += "\n[تسجيل صوتي مرفق]"
    history.extend([{"role": "user", "content": visible}, {"role": "assistant", "content": answer}])
    return history, "", None, None, "تم."


def colab_new_chat():
    global COLAB_CONVERSATION_ID
    COLAB_CONVERSATION_ID = create_new_conversation(COLAB_USER_ID)
    return [], "بدأت محادثة جديدة."

with gr.Blocks(title="Dr-Zyren") as colab_chat_app:
    gr.Markdown("## محادثة Dr-Zyren")
    chat_window = gr.Chatbot(height=440, label="المحادثة")
    message_box = gr.Textbox(placeholder="اكتب رسالتك…", label="رسالة", lines=2)
    with gr.Row():
        chat_image = gr.Image(type="filepath", sources=["upload", "clipboard"], label="صورة")
        chat_audio = gr.Audio(type="filepath", sources=["upload", "microphone"], label="تسجيل صوتي")
    with gr.Row():
        send_button = gr.Button("إرسال", variant="primary")
        new_chat_button = gr.Button("محادثة جديدة")
    chat_status = gr.Markdown()
    inputs = [message_box, chat_image, chat_audio, chat_window]
    outputs = [chat_window, message_box, chat_image, chat_audio, chat_status]
    send_button.click(colab_chat_send, inputs, outputs)
    message_box.submit(colab_chat_send, inputs, outputs)
    new_chat_button.click(colab_new_chat, outputs=[chat_window, chat_status])

print("Colab chat ready.")
colab_chat_app.launch(share=False, inline=True)

Colab chat ready.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

### 23 | Start Telegram


In [33]:
if not COHERE_MODEL_TEST_OK:
    raise RuntimeError(
        "Cohere API test failed. Resolve the Cohere API error before starting Telegram."
    )

if application.running:

    print("Dr-Zyren is already running.")

else:

    await application.initialize()
    await application.start()

    await application.updater.start_polling(
        drop_pending_updates=True,
        allowed_updates=["message", "edited_message"]
    )

    print("Dr-Zyren Telegram bot is RUNNING inside Colab.")
    print("Run the Stop Telegram Bot step below to stop polling.")

Dr-Zyren Telegram bot is RUNNING inside Colab.
Run the Stop Telegram Bot step below to stop polling.


### 24 | Stop Telegram


In [ ]:
try:

    if application.updater and application.updater.running:
        await application.updater.stop()

    if application.running:
        await application.stop()
        await application.shutdown()

    await model_client.close()

    print("Dr-Zyren Telegram polling stopped.")
    print("The bot will not process new messages.")
    print("SQLite data and reference cases remain saved.")
    print("Selected Cohere client closed.")

except Exception as e:

    print("Shutdown note:", type(e).__name__, str(e)[:500])


### 25 | Backup


In [ ]:
import zipfile
backup_path = PROJECT_DIR / f"Dr-Zyren-backup-{datetime.now().strftime('%Y%m%d-%H%M%S')}.zip"
with zipfile.ZipFile(backup_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    if DB_PATH.exists():
        zf.write(DB_PATH, arcname="data/dr_zyren.sqlite3")
    if CASE_IMAGE_DIR.exists():
        for f in CASE_IMAGE_DIR.rglob("*"):
            if f.is_file():
                zf.write(f, arcname=str(Path("data/reference_cases") / f.relative_to(CASE_IMAGE_DIR)))
print("Backup created:", backup_path)